# 스케일링·정규화 방법론 비교

현재 학습·실시간 추론이 공유하는 `sample_standard_v1`을 기준으로 상대가격, robust z-score, min-max 후보를 비교한다. 이 단계는 후보를 제거하는 screening이며 최종 채택은 동일 symbol split의 validation 성능으로 결정한다.

## 누수 방지 계약

- 샘플 단위 변환은 해당 시퀀스 내부 값만 사용한다.
- 데이터셋 전체 통계를 사용하는 변환은 train symbol에서만 적합하고 validation/test에는 고정 적용해야 한다.
- 실시간 추론은 학습과 동일한 변환 함수·피처 순서를 사용해야 한다.
- 이 노트북은 원천 parquet와 샘플을 변경하거나 Supabase에 저장하지 않는다.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from pivot.config import PreprocessPreset, Timeframe
from pivot.dataset.build import run_preprocess
from pivot.dataset.transforms import sample_standardize
from pivot.ingestion.cache import cache_path, load_cache
from pivot.ingestion.fetch import cache_broker

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

SYMBOL = '005930'
TIMEFRAME = Timeframe.from_code('day')
MAX_SAMPLES = 1000
PRESET = PreprocessPreset(name='scaling-research', timeframe=TIMEFRAME)


In [ ]:
path = cache_path(ROOT / 'data', cache_broker(), TIMEFRAME.code, SYMBOL)
frame = load_cache(path)
if frame is None or frame.empty:
    raise FileNotFoundError(f'{SYMBOL} {TIMEFRAME.code} 캐시가 없습니다: {path}')

result = run_preprocess(frame, PRESET)
feature_columns = result.feature_columns
selected_samples = result.samples[:MAX_SAMPLES]
sequences = [
    result.frame.iloc[sample.start_position : sample.end_position + 1]
    [feature_columns].to_numpy(dtype=np.float32)
    for sample in selected_samples
]
labels = np.asarray([sample.label for sample in selected_samples], dtype=np.int64)

pd.Series([len(sequence) for sequence in sequences], name='sequence_length').describe()

In [ ]:
PRICE_COLUMNS = {'Open', 'High', 'Low', 'Close'}
price_indexes = [
    index
    for index, column in enumerate(feature_columns)
    if column in PRICE_COLUMNS or column.isdigit()
]
other_indexes = [
    index for index in range(len(feature_columns)) if index not in price_indexes
]
close_index = feature_columns.index('Close')

def minmax_sample(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=np.float32)
    low = values.min(axis=0, keepdims=True)
    span = values.max(axis=0, keepdims=True) - low
    safe_span = np.where(span > 0, span, 1.0)
    return ((values - low) / safe_span).astype(np.float32)

def robust_sample(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=np.float32)
    median = np.median(values, axis=0, keepdims=True)
    q1, q3 = np.quantile(values, [0.25, 0.75], axis=0, keepdims=True)
    iqr = q3 - q1
    safe_iqr = np.where(iqr > 0, iqr, 1.0)
    return ((values - median) / safe_iqr).astype(np.float32)

def relative_to_last_close(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=np.float32)
    transformed = np.zeros_like(values)
    anchor = float(values[-1, close_index])
    if not np.isfinite(anchor) or anchor == 0:
        raise ValueError('last close must be finite and non-zero')
    transformed[:, price_indexes] = values[:, price_indexes] / anchor - 1.0
    if other_indexes:
        transformed[:, other_indexes] = sample_standardize(
            np.log1p(np.maximum(values[:, other_indexes], 0.0))
        )
    return transformed

TRANSFORMS = {
    'sample_standard_v1': sample_standardize,
    'relative_last_close_v1': relative_to_last_close,
    'robust_iqr_v1': robust_sample,
    'minmax_sample_v1': minmax_sample,
}


In [ ]:
def transform_diagnostics(name: str, transform) -> dict:
    transformed = [transform(sequence) for sequence in sequences]
    flat = np.concatenate(transformed, axis=0)
    per_sample_means = np.vstack([item.mean(axis=0) for item in transformed])
    per_sample_stds = np.vstack([item.std(axis=0) for item in transformed])
    return {
        'method': name,
        'finite_pct': np.isfinite(flat).mean(),
        'mean_abs': np.abs(flat).mean(),
        'p99_abs': np.quantile(np.abs(flat), 0.99),
        'outlier_abs_gt_5_pct': (np.abs(flat) > 5).mean(),
        'sample_mean_drift': np.abs(per_sample_means).mean(),
        'sample_std_mean': per_sample_stds.mean(),
        'constant_feature_pct': (per_sample_stds < 1e-8).mean(),
    }

diagnostics = pd.DataFrame(
    [transform_diagnostics(name, transform) for name, transform in TRANSFORMS.items()]
).set_index('method')
display(diagnostics.style.format({
    'finite_pct': '{:.2%}',
    'outlier_abs_gt_5_pct': '{:.2%}',
    'constant_feature_pct': '{:.2%}',
}))

In [ ]:
# 간단한 nearest-centroid screening. 최종 모델 성능 지표가 아니다.
def embedding(sequence: np.ndarray) -> np.ndarray:
    return np.concatenate([
        sequence.mean(axis=0),
        sequence.std(axis=0),
        sequence.min(axis=0),
        sequence.max(axis=0),
        sequence[-1],
    ])

def centroid_accuracy(transform) -> float:
    features = np.vstack([embedding(transform(sequence)) for sequence in sequences])
    boundary = max(int(len(features) * 0.7), 1)
    train_x, test_x = features[:boundary], features[boundary:]
    train_y, test_y = labels[:boundary], labels[boundary:]
    mean = train_x.mean(axis=0, keepdims=True)
    std = train_x.std(axis=0, keepdims=True)
    std[std == 0] = 1.0
    train_x = (train_x - mean) / std
    test_x = (test_x - mean) / std
    classes = np.unique(train_y)
    centroids = np.vstack([train_x[train_y == label].mean(axis=0) for label in classes])
    distances = ((test_x[:, None, :] - centroids[None, :, :]) ** 2).mean(axis=2)
    predicted = classes[distances.argmin(axis=1)]
    return float((predicted == test_y).mean())

screening = pd.Series(
    {name: centroid_accuracy(transform) for name, transform in TRANSFORMS.items()},
    name='chronological_centroid_accuracy',
).sort_values(ascending=False)
screening

In [ ]:
# 한 샘플의 변환 전후 수치를 피처별로 검수한다.
SAMPLE_INDEX = min(10, len(sequences) - 1)
raw = sequences[SAMPLE_INDEX]
comparison = {}
for name, transform in TRANSFORMS.items():
    value = transform(raw)
    comparison[name] = pd.DataFrame(value, columns=feature_columns).describe().T
display(pd.concat(comparison, names=['method', 'feature']))

## 채택 기준과 후속 작업

1. NaN/Inf가 생기는 방법은 제외한다.
2. 극단값과 상수 피처 비율을 확인한다.
3. 후보별 데이터셋을 만들지 말고, loader transform을 실험 옵션으로 추가한 뒤 동일 shard를 사용한다.
4. 동일 symbol split, model, seed에서 validation macro F1과 클래스별 precision을 비교한다.
5. 채택된 transform은 training과 live가 공유하는 `pivot.dataset.transforms`에 구현하고 run snapshot에 버전을 저장한다.